In [1]:
import immrax as irx
import jax
import jax.numpy as jnp

class Sys (irx.System) :
    def __init__ (self) :
        self.evolution = 'continuous'
        self.xlen = 5
    def f (self, t, x, u) :
        x, y, vx, vy, theta = x.ravel()
        u1, u2 = u.ravel()
        g = 9.81
        return jnp.array([
            vx,
            vy,
            -1*u1*jnp.sin(theta),
            u1*jnp.cos(theta) - g,
            u2
        ])

sys = Sys()

An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.


In [12]:
jacM = irx.jacM(sys.f)
mjacM = irx.mjacM(sys.f)

it = irx.interval(0.)
ix = irx.icentpert(jnp.zeros(5), 0.25)
iu = irx.icentpert(jnp.zeros(2), 1)
mMt, mMx, mMu = mjacM(it, ix, iu, permutations=irx.Permutation((0,6,7,1,2,3,4,5)))[0]
jMt, jMx, jMu = jacM(it, ix, iu)

print(f'{mMx=}')
print(f'{mMu=}')
print(f'{jMx=}')

K = jnp.array([[-1., -1., -1., -1., -1.], [-1., -1., -1., -1., -1.]])
Mx_c = irx.get_sparse_corners(mMx)
# Mx_c = irx.get_sparse_corners(jMx)
print(len(Mx_c))

mMx=[[ 0.          0.          1.          0.          0.        ]
 [ 0.          0.          0.          1.          0.        ]
 [ 0.          0.          0.          0.         -1.        ]
 [ 0.          0.          0.          0.         -0.24740396]
 [ 0.          0.          0.          0.          0.        ]] <= x <= [[0.         0.         1.         0.         0.        ]
 [0.         0.         0.         1.         0.        ]
 [0.         0.         0.         0.         1.        ]
 [0.         0.         0.         0.         0.24740396]
 [0.         0.         0.         0.         0.        ]]
mMu=[[0. 0.]
 [0. 0.]
 [0. 0.]
 [1. 0.]
 [0. 1.]] <= x <= [[0. 0.]
 [0. 0.]
 [0. 0.]
 [1. 0.]
 [0. 1.]]
jMx=[[ 0.          0.          1.          0.          0.        ]
 [ 0.          0.          0.          1.          0.        ]
 [ 0.          0.          0.          0.         -1.        ]
 [ 0.          0.          0.          0.         -0.24740396]
 [ 0.          0.    

In [20]:
import cvxpy as cp
import numpy as np

print(Mx_c)
Mu = mMu.lower

P = cp.Variable((5,5), symmetric=True)
# P = cp.Parameter((3,3), symmetric=True)
# P.value = np.array([[1., 0., 0.], [0., 1., 0.], [0., 0., 1.]])
# c = cp.Variable(1)
c = 0.8
eps = 0.01
cons = [P >> eps]
cons.extend([(Mx_c[i] + Mu@K).T @ P + P @ (Mx_c[i] + Mu@K) << 2*c*P for i in range(len(Mx_c))])

prob = cp.Problem(cp.Minimize(c), cons)
# prob.solve(verbose=True)
prob.solve()
print(prob.status)
print(P.value)
P = P.value
# print(c.value)

for i in range(len(Mx_c)):
    print(Mx_c[i].T @ P + P @ Mx_c[i])

eigs = jnp.real(jnp.linalg.eigvals(P))
print(eigs)
print(jnp.linalg.eig(P))

# print(jnp.linalg.eigvals(Jtest@P + P@Jtest.T))


[Array([[ 0.        ,  0.        ,  1.        ,  0.        ,  0.        ],
       [ 0.        ,  0.        ,  0.        ,  1.        ,  0.        ],
       [ 0.        ,  0.        ,  0.        ,  0.        , -1.        ],
       [ 0.        ,  0.        ,  0.        ,  0.        , -0.24740396],
       [ 0.        ,  0.        ,  0.        ,  0.        ,  0.        ]],      dtype=float32), Array([[ 0.        ,  0.        ,  1.        ,  0.        ,  0.        ],
       [ 0.        ,  0.        ,  0.        ,  1.        ,  0.        ],
       [ 0.        ,  0.        ,  0.        ,  0.        , -1.        ],
       [ 0.        ,  0.        ,  0.        ,  0.        ,  0.24740396],
       [ 0.        ,  0.        ,  0.        ,  0.        ,  0.        ]],      dtype=float32), Array([[ 0.        ,  0.        ,  1.        ,  0.        ,  0.        ],
       [ 0.        ,  0.        ,  0.        ,  1.        ,  0.        ],
       [ 0.        ,  0.        ,  0.        ,  0.        ,  1.    